<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 8 — The Admissibility-Capacity Ledger
## Reviewer Walkthrough — Interactive Mathematical Derivation

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.PENDING.svg)](https://doi.org/10.5281/zenodo.PENDING) [![GitHub](https://img.shields.io/badge/GitHub-APF--Paper--8--Admissibility--Capacity--Ledger-blue?logo=github)](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)

This notebook is a **self-contained verification** of every theorem in Paper 8 — The Admissibility-Capacity Ledger. Each theorem gets a markdown cell (formula, proof sketch, dependencies) followed by a live code cell that executes the check and unpacks the mathematical result.

**The claim:** the 36 theorems below are all [P]-derived from Axiom A1 (finite information capacity) plus PLEC's structural components. No free parameters are introduced at any step.

**To run end-to-end:** *Runtime → Run all* (or `Cmd/Ctrl + F9`). Total verification takes under a minute.

**Codebase:** APF v6.9 — 355 `verify_all` checks across 342 bank-registered theorems in 19 modules, 48 quantitative predictions, **0 free parameters**. This repo bundles the **36-check subset** directly cited by Paper 8.

---

### The APF corpus

Paper 8 is one companion in a 9-paper series. The full set:

| # | Title | Zenodo DOI | GitHub repo | Status |
|---|---|---|---|---|
| 0 | What Physics Permits | [10.5281/zenodo.18605692](https://doi.org/10.5281/zenodo.18605692) | [`APF-Paper-0-What-Physics-Permits`](https://github.com/Ethan-Brooke/APF-Paper-0-What-Physics-Permits) | public |
| 1 | The Enforceability of Distinction | [10.5281/zenodo.18604678](https://doi.org/10.5281/zenodo.18604678) | [`APF-Paper-1-The-Enforceability-of-Distinction`](https://github.com/Ethan-Brooke/APF-Paper-1-The-Enforceability-of-Distinction) | public |
| 2 | The Structure of Admissible Physics | [10.5281/zenodo.18604839](https://doi.org/10.5281/zenodo.18604839) | [`APF-Paper-2-The-Structure-of-Admissible-Physics`](https://github.com/Ethan-Brooke/APF-Paper-2-The-Structure-of-Admissible-Physics) | public |
| 3 | Ledgers | [10.5281/zenodo.18604844](https://doi.org/10.5281/zenodo.18604844) | [`APF-Paper-3-Ledgers-Entropy-Time-Cost`](https://github.com/Ethan-Brooke/APF-Paper-3-Ledgers-Entropy-Time-Cost) | public |
| 4 | Admissibility Constraints and Structural Saturation | [10.5281/zenodo.18604845](https://doi.org/10.5281/zenodo.18604845) | [`APF-Paper-4-Admissibility-Constraints-Field-Content`](https://github.com/Ethan-Brooke/APF-Paper-4-Admissibility-Constraints-Field-Content) | public |
| 5 | Quantum Structure from Finite Enforceability | [10.5281/zenodo.18604861](https://doi.org/10.5281/zenodo.18604861) | [`APF-Paper-5-Quantum-Structure-Hilbert-Born`](https://github.com/Ethan-Brooke/APF-Paper-5-Quantum-Structure-Hilbert-Born) | public |
| 6 | Dynamics and Geometry as Optimal Admissible Reallocation | [10.5281/zenodo.18604874](https://doi.org/10.5281/zenodo.18604874) | [`APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity`](https://github.com/Ethan-Brooke/APF-Paper-6-Dynamics-Geometry-Spacetime-Gravity) | public |
| 7 | Action, Internalization, and the Lagrangian | [10.5281/zenodo.18604875](https://doi.org/10.5281/zenodo.18604875) | [`APF-Paper-7-Action-Internalization-Lagrangian`](https://github.com/Ethan-Brooke/APF-Paper-7-Action-Internalization-Lagrangian) | public |
| 13 | The Minimal Admissibility Core | [10.5281/zenodo.18614663](https://doi.org/10.5281/zenodo.18614663) | [`APF-Paper-13-The-Minimal-Admissibility-Core`](https://github.com/Ethan-Brooke/APF-Paper-13-The-Minimal-Admissibility-Core) | public |
| — | Canonical codebase (v6.9) | [10.5281/zenodo.18604548](https://doi.org/10.5281/zenodo.18604548) | [`APF-Codebase`](https://github.com/Ethan-Brooke/APF-Codebase) | pending |

The canonical engine (full 342-theorem bank) is at [APF-Codebase](https://doi.org/10.5281/zenodo.18604548).


## Setup

Clone the paper-companion repo and install the bundled `apf/` package. This gives us access to the registry of check functions and the utility helpers for exact rational arithmetic.

In [ ]:
# Clone the paper-companion repo (skip if already cloned in this runtime)
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger.git 2>/dev/null || true
import os
if os.path.isdir('APF-Paper-8-Admissibility-Capacity-Ledger'):
    os.chdir('APF-Paper-8-Admissibility-Capacity-Ledger')
!pip install -e . --quiet 2>&1 | tail -1

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from fractions import Fraction
import inspect
import json

print(f'Loaded {len(REGISTRY)} checks from this repo (expected 36).')
print('Exact rational arithmetic is available via `from fractions import Fraction`.')
print('Pretty-print helper defined below for consistent result display.')

In [ ]:
# Pretty-print helper: surfaces ALL mathematical content from a result dict,
# not just a pass/fail flag.
def show(check_name):
    """Run a check and surface its full mathematical result."""
    fn = get_check(check_name)
    r = fn()
    print(f'━━━ {check_name} ━━━')
    if not isinstance(r, dict):
        print(f'  result: {r}'); return r
    status = 'PASS' if r.get('passed', True) else 'FAIL'
    print(f'  status: {status}')
    # Surface mathematical content in a stable order
    for k in ('key_result', 'result', 'value', 'theorem', 'witnesses',
              'intermediate', 'derivation', 'formula', 'predicted',
              'observed', 'error_sigma', 'depends_on', 'module'):
        if k in r:
            val = r[k]
            s = str(val)
            if len(s) > 200: s = s[:197] + '...'
            print(f'  {k}: {s}')
    # Surface anything else not already covered
    seen = {'passed', 'key_result', 'result', 'value', 'theorem',
            'witnesses', 'intermediate', 'derivation', 'formula',
            'predicted', 'observed', 'error_sigma', 'depends_on', 'module'}
    for k, v in r.items():
        if k in seen: continue
        s = str(v)
        if len(s) > 200: s = s[:197] + '...'
        print(f'  {k}: {s}')
    return r

def show_source(check_name, max_lines=30):
    """Show the first N lines of a check function's source (for transparency)."""
    fn = get_check(check_name)
    try:
        src = inspect.getsource(fn)
    except OSError:
        print('  (source unavailable)')
        return
    lines = src.split('\n')[:max_lines]
    for ln in lines:
        print(ln)
    if len(src.split('\n')) > max_lines:
        print(f'    ... ({len(src.split(chr(10)))-max_lines} more lines)')

print('Helpers ready: show(<name>), show_source(<name>).')

## What this paper claims

The Admissibility-Capacity Ledger: a unifying record (K, d_eff) read through six regime projections (pi_T, pi_G, pi_Q, pi_F, pi_C, pi_A) with four consistency identities I_1-I_4. Paper 8's only non-trivial structural claim is the gauge-cosmological bridge I_2 (Theorem 1.1 of the Technical Supplement Formal Kernel): a unique G_SM-invariant 42-dimensional subspace V_Lambda of V_61 exists under the Standard-Model gauge action and T12 partition constraint, inducing the residual partition 3+16+42=61 and cosmological fraction Omega_Lambda = 42/61. I_1 is definitional under the cell-count convention K_horizon = A/(4 l_P^2); I_3 and I_4 are standard finite-dimensional closures. Supplement contains self-contained proof via Maschke semisimplicity + minimal working example at toy interface K=3 d_eff=4 auditable in ≤10 lines numpy + Planck 2018 multivariate Bayes-factor restricted sanity check + finite-volume C*-algebra formulation with Type III_1 classification at infinite volume. Four Phase 16-20 reviewer-response passes culminated in v2.9 main + v2.5 supplement.

What follows is the theorem-by-theorem derivation, grouped by tier (axioms → lemmas → theorems → predictions). Every cell you run is a live check from the canonical bank.

---

## Tier 1 · Structural Lemmas

The four structural lemmas that bridge axioms to theorems. Each is proved directly from A1 + PLEC components; nothing external is imported.


### 1. `check_L_Omega_Lambda_is_entropy_fraction` — L_Omega_Lambda_is_entropy_fraction

**Statement (from codebase):** L_Omega_Lambda_is_entropy_fraction [P] — the dark-energy eureka.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Omega_Lambda_is_entropy_fraction')

### 2. `check_L_Omega_b_is_entropy_fraction` — L_Omega_b_is_entropy_fraction

**Statement (from codebase):** L_Omega_b_is_entropy_fraction [P] — Baryon fraction as entropy allocation.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Omega_b_is_entropy_fraction')

### 3. `check_L_Omega_c_is_entropy_fraction` — L_Omega_c_is_entropy_fraction

**Statement (from codebase):** L_Omega_c_is_entropy_fraction [P] — CDM fraction as entropy allocation.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Omega_c_is_entropy_fraction')

### 4. `check_L_Lambda_absolute_numerical_formula` — L_Lambda_absolute_numerical_formula

**Statement (from codebase):** L_Lambda_absolute_numerical_formula [P] — APF coefficient formula matches Λ.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Lambda_absolute_numerical_formula')

### 5. `check_L_Sigma_m_nu_suggestive` — L_Sigma_m_nu_suggestive

**Statement (from codebase):** L_Sigma_m_nu_suggestive [C, open] — neutrino mass sum consistent with k=15.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_Sigma_m_nu_suggestive')

### 6. `check_L_eta_does_not_fit_cleanly` — L_eta_does_not_fit_cleanly

**Statement (from codebase):** L_eta_does_not_fit_cleanly [C, open] — baryon-to-photon ratio scope.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_eta_does_not_fit_cleanly')

### 7. `check_L_horizon_convention_equivalence` — L_horizon_convention_equivalence

**Statement (from codebase):** L_horizon_convention_equivalence [P] — Conventions A and B agree.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_L_horizon_convention_equivalence')

---

## Tier 2 · Derived Theorems

The paper's main theorems. Each depends on lemmas and/or earlier theorems as noted in the `depends_on` chain. Epistemic tags: **[P]** = internally derived; **[P]+[I]** = APF prepares the admissible class, then an external mathematical theorem (Solèr, Gleason, HKM, Malament, Lovelock) closes the classification.


### 8. `check_T_ACC_unification` — T_ACC_unification

**Statement (from codebase):** T_ACC_unification: Admissibility-Capacity Ledger unification [P].

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_ACC_unification')

### 9. `check_T_three_level_unification` — T_three_level_unification

**Statement (from codebase):** T_three_level_unification — Three-level refinement of T_ACC [P]. (tier 4)

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_three_level_unification')

### 10. `check_T_subspace_functors_unified` — T_subspace_functors_unified

**Statement (from codebase):** T_subspace_functors_unified [P] — All three 14c functors delivered. (tier 4)

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_subspace_functors_unified')

### 11. `check_T_fractional_reading_equivalence` — T_fractional_reading_equivalence

**Statement (from codebase):** T_fractional_reading_equivalence [P] — FRE at any ACC interface.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_fractional_reading_equivalence')

### 12. `check_T_residual_entropy_closure` — T_residual_entropy_closure

**Statement (from codebase):** T_residual_entropy_closure [P] — Entropy fractions sum to 1.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_residual_entropy_closure')

### 13. `check_T_FRE_SM_to_entropy_dictionary` — T_FRE_SM_to_entropy_dictionary

**Statement (from codebase):** T_FRE_SM_to_entropy_dictionary [P] — Full SM entropy dictionary. (tier 4)

**Depends on:** `check_T_fractional_reading_equivalence`, `check_L_Omega_b_is_entropy_fraction`, `check_L_Omega_c_is_entropy_fraction`, `check_L_Omega_Lambda_is_entropy_fraction`, `check_T_residual_entropy_closure`

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_FRE_SM_to_entropy_dictionary')

### 14. `check_T_Lambda_coefficient_degeneracy_audit` — T_Lambda_coefficient_degeneracy_audit

**Statement (from codebase):** T_Lambda_coefficient_degeneracy_audit [C] — Numerical vs structural privilege.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_Lambda_coefficient_degeneracy_audit')

### 15. `check_T_Lambda_absolute_extended_formula` — T_Lambda_absolute_extended_formula

**Statement (from codebase):** T_Lambda_absolute_extended_formula [P] — Universal cosmological density formula.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_Lambda_absolute_extended_formula')

### 16. `check_T_Lambda_to_H0_inversion` — T_Lambda_to_H0_inversion

**Statement (from codebase):** T_Lambda_to_H0_inversion [P] — APF prediction for the Hubble constant.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_Lambda_to_H0_inversion')

### 17. `check_T_T_CMB_absolute_formula` — T_T_CMB_absolute_formula

**Statement (from codebase):** T_T_CMB_absolute_formula [P] — APF predicts T_CMB via thermal C/d_eff^k.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_T_CMB_absolute_formula')

### 18. `check_T_Lambda_absolute_bulletproof` — T_Lambda_absolute_bulletproof

**Statement (from codebase):** T_Lambda_absolute_bulletproof [P] — Full robustness of Λ-absolute claim.

**Depends on:** `check_L_Lambda_absolute_numerical_formula`, `check_T_Lambda_absolute_extended_formula`, `check_T_Lambda_coefficient_degeneracy_audit`

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_Lambda_absolute_bulletproof')

### 19. `check_T_Lambda_partition_function_at_beta_zero` — T_Lambda_partition_function_at_beta_zero

**Statement (from codebase):** T_Lambda_partition_function_at_beta_zero [P] — Operator-level β → 0 limit.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_Lambda_partition_function_at_beta_zero')

### 20. `check_T_Lambda_d2_operator_derivation` — T_Lambda_d2_operator_derivation

**Statement (from codebase):** T_Lambda_d2_operator_derivation [P over [P]+[P_structural]+[C]] —

**Depends on:** `check_T_Lambda_partition_function_at_beta_zero`

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_Lambda_d2_operator_derivation')

### 21. `check_T_I3_operator_self_identification` — T_I3_operator_self_identification

**Statement (from codebase):** T_I3_operator_self_identification [P_comp] — Composed top for I3.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_I3_operator_self_identification')

### 22. `check_T_I4_operator_self_identification` — T_I4_operator_self_identification

**Statement (from codebase):** T_I4_operator_self_identification [P_comp] — Composed top for I4.

**Depends on:** `check_T_Lambda_partition_function_at_beta_zero`

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_I4_operator_self_identification')

### 23. `check_T_bridge_observer_independence_open` — T_bridge_observer_independence_open

**Statement (from codebase):** T_bridge_observer_independence_open [C, open] —

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('check_T_bridge_observer_independence_open')

---

## Supporting Results

Supporting results that don't fit the tier scheme cleanly — utility identities, red-team checks, internal consistency tests.


### 24. `ACC` — ACC

**Statement (from codebase):** 

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('ACC')

### 25. `pi_T` — pi_T

**Statement (from codebase):** Thermodynamic projection: S(rho, Gamma) = ACC(rho, Gamma).

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('pi_T')

### 26. `pi_G` — pi_G

**Statement (from codebase):** Gravitational projection: S_BH = A/(4 ell_P^2) = ACC_horizon.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('pi_G')

### 27. `pi_Q` — pi_Q

**Statement (from codebase):** Quantum projection: dim H(rho, Gamma) = exp(ACC) = N.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('pi_Q')

### 28. `pi_F` — pi_F

**Statement (from codebase):** Gauge/field projection: K(rho, Gamma) — the structural capacity.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('pi_F')

### 29. `pi_C` — pi_C

**Statement (from codebase):** Cosmological projection: residuals as fractions over K.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('pi_C')

### 30. `pi_A` — pi_A

**Statement (from codebase):** Action projection: Z(beta) = sum_g exp(-beta H(g)).

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('pi_A')

### 31. `_identity_I1_holographic` — _identity_I1_holographic

**Statement (from codebase):** Identity I1 (Holographic): S_BH = ln(dim H_horizon) = ACC_horizon.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('_identity_I1_holographic')

### 32. `_identity_I2_gauge_cosmological` — _identity_I2_gauge_cosmological

**Statement (from codebase):** Identity I2 (Gauge-cosmological): K in pi_F equals denominator in pi_C.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('_identity_I2_gauge_cosmological')

### 33. `_identity_I3_thermo_quantum` — _identity_I3_thermo_quantum

**Statement (from codebase):** Identity I3 (Thermo-quantum): S_vN(rho_max_mixed) = ln(dim H) = ACC.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('_identity_I3_thermo_quantum')

### 34. `_identity_I4_action_thermo` — _identity_I4_action_thermo

**Statement (from codebase):** Identity I4 (Action-thermo): lim_{beta -> 0} ln Z(beta) = ACC.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('_identity_I4_action_thermo')

### 35. `_identity_composition_tests` — _identity_composition_tests

**Statement (from codebase):** 

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('_identity_composition_tests')

### 36. `_build_model` — _build_model

**Statement (from codebase):** Build the (H_micro, H_total, P_vac_slot0) tuple for a model interface.

The cell below runs the check and unpacks every field in its result dict — `key_result`, `witnesses`, `depends_on`, and any intermediate mathematical quantity the check surfaces. The pass/fail flag is one line; the real content is everything else.

In [ ]:
show('_build_model')

---

## Full scorecard

Aggregate verdict across all 36 bundled checks. If all pass, the claims in Paper 8 are computationally verified against the canonical APF bank at codebase v6.9.

For the full 342-theorem bank, clone the canonical engine ([https://doi.org/10.5281/zenodo.18604548](https://doi.org/10.5281/zenodo.18604548)).

In [ ]:
results = run_all(verbose=False)
passed = sum(1 for r in results if r.get('passed', True))
failed = len(results) - passed
print(f'━━━ Paper 8 scorecard ━━━')
print(f'Bundled checks: {len(results)}')
print(f'Passed:         {passed}')
print(f'Failed:         {failed}')
if failed:
    print()
    print('FAILED:')
    for r in results:
        if not r.get('passed', True):
            print(f"  - {r.get('name', '?')}: {r.get('error', r.get('reason', '?'))}")
else:
    print()
    print('All checks pass. This is not proof of the framework — it is proof')
    print('that the claims of Paper {0} are internally consistent with the'.format(8))
    print('canonical APF computational engine. The framework itself is')
    print('defended via audit, not assertion. See ai_context/AUDIT_DISCIPLINE.md.')

---

## Next steps

- **Full derivation DAG:** [interactive visualization](https://ethan-brooke.github.io/APF-Paper-8-Admissibility-Capacity-Ledger/) (hover theorems for dependencies, click to focus subgraphs).
- **Reviewers' guide:** [`REVIEWERS_GUIDE.md`](REVIEWERS_GUIDE.md) — physics-first 9-section walkthrough.
- **AI onboarding:** [`START_HERE.md`](START_HERE.md) — operational checklist.
- **Machine-readable repo map:** [`ai_context/repo_map.json`](ai_context/repo_map.json) — every file with purpose, audience, references.
- **Theorem catalog:** [`ai_context/theorems.json`](ai_context/theorems.json) — 342-entry bank with tags and dependencies.
- **The paper:** `.pdf` in this repo.
- **Cite:** [10.5281/zenodo.PENDING](https://doi.org/10.5281/zenodo.PENDING).

---

*Notebook generated by the APF `create-repo` skill. Codebase: v6.9 (355 `verify_all` checks, 342 bank-registered theorems across 19 modules, 48 quantitative predictions, 0 free parameters).*